# 实践项目 06：空间细胞通讯候选关系

我们使用小鼠肾脏空间转录组结果，检查细胞位置、细胞类型、配体—受体和候选通讯方向。配套研究使用三层图网络训练。本次实践保留读懂数据与结果所需的关键步骤，可以在普通电脑上完成。

Kaggle 是首选运行环境。把题目 Notebook 复制到自己的账户并添加配套数据集，按顺序完成标有“需要完成”的单元格。

## 任务总览

1. 核对细胞表和候选边表，查看 H&E 与细胞坐标。
2. 筛选 `Vegfa-Flt1/Kdr`，计算距离—共表达分数。
3. 按发送和接收细胞类型汇总有方向的候选关系。
4. 绘制空间候选边并结合遮边测试指标解释这些候选关系的含义。

候选分数用于安排后续验证的优先次序：RNA 表达提供配体和受体的转录信号，空间位置说明细胞是否邻近；蛋白结合和通路激活需要对应的蛋白测量或扰动实验。


In [ ]:
from pathlib import Path  # 处理 Kaggle 与本地文件路径
import json  # 读取并保存结果摘要
import numpy as np  # 进行数组与数值计算
import pandas as pd  # 读取细胞表和候选通讯边表
import matplotlib.pyplot as plt  # 绘制组织、热图和空间关系


def find_file(name):  # 在 Kaggle 数据集或本地项目目录中寻找指定文件
    roots = [Path('/kaggle/input'), Path.cwd(), Path.cwd() / '数据', Path.cwd() / 'data', Path.cwd().parent / '数据', Path.cwd().parent / 'data']  # 同时兼容项目根目录和实践/答案子目录
    for root in roots:  # 遍历可能的数据根目录
        if not root.exists():  # Kaggle 之外的环境可能没有 /kaggle/input
            continue  # 跳过不存在的目录
        matches = list(root.rglob(name))  # 递归查找文件，兼容 Kaggle 自动添加的数据集目录名
        if matches:  # 找到后立即返回第一项
            return matches[0]  # 返回实际文件路径
    raise FileNotFoundError(f'没有找到 {name}，请确认已添加项目数据集。')  # 给出可操作的错误信息

LOCAL_PROJECT = Path.cwd().parent if (Path.cwd().parent / '数据').exists() else Path.cwd()  # 从实践/答案子目录运行时回到项目根目录
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else LOCAL_PROJECT / '结果'  # Kaggle 与本地共用输出规则
OUT.mkdir(parents=True, exist_ok=True)  # 确保结果目录存在


cells = pd.read_csv(find_file('ccc_cells.csv'))  # 每行是一枚细胞，包含坐标与细胞类型
edges = pd.read_csv(find_file('ccc_edges.csv'))  # 每行是一条有方向的候选配体—受体边
history = pd.read_csv(find_file('ccc_training_history.csv'))  # 配套图网络的训练记录
metrics = json.loads(find_file('ccc_model_metrics.json').read_text(encoding='utf-8'))  # 配套遮边测试指标
source_manifest = json.loads(find_file('source_manifest.json').read_text(encoding='utf-8'))  # 读取原图与预览图的尺寸关系
he_image = plt.imread(find_file('ccc_kidney_he.png'))  # 读取肾脏 H&E 预览图
print('cells:', cells.shape, 'edges:', edges.shape)  # 先确认数据已经成功加载


## 任务 1：细胞与候选边核对

输出细胞数、细胞类型数、候选边数、配体—受体对的数量和低置信度注释比例，并把细胞坐标叠加到 H&E 预览图。观察坐标范围与图像范围是否相符。

配套候选表按图网络分数 `gnn_score` 筛选：在 84,792 条关系中，保留分数不低于第 80 百分位阈值的 16,959 条，并按分数降序排列。`lr_pairs` 统计其中不同配体—受体组合的数量。


In [ ]:
# ===== 项目06·任务1·需要完成（开始） =====
# TODO：生成 summary，并绘制 H&E 与细胞坐标叠加图。
summary = None  # 保存数据核对结果
# ===== 项目06·任务1·需要完成（结束） =====
print(summary)  # 显示核对结果


## 任务 2：Vegfa–Flt1/Kdr 候选证据

保留发送配体为 `Vegfa`、接收受体为 `Flt1` 或 `Kdr` 的候选边，计算 `log1p(coexpression) × exp(-distance/120)`，再与配套 `evidence_score` 比较排序。

- `coexpression`：发送细胞的配体表达值与接收细胞的受体表达值之积。两个表达值都经过每细胞总量归一化和 `log1p` 变换。
- `distance`：两细胞在原始 `X/Y` 坐标中的欧氏距离，沿用数据表的坐标单位。`120` 是同一单位下的距离衰减尺度；距离等于 120 时，距离权重为 `exp(-1)`，约为 0.368。
- `evidence_score`：配套预处理对 `log1p(coexpression)` 按中位数和绝对偏差中位数（MAD）做稳健标准化，经 sigmoid 映射后乘距离权重，最后限制在 `[0,1]`。它与本任务的简化分数共用表达和距离信息。

Spearman 系数比较两套分数的排序是否相近，数值越接近 1 表示排序越一致。两套分数的具体数值范围不同，散点图用于观察高低排序与差异。


In [ ]:
# ===== 项目06·任务2·需要完成（开始） =====
# TODO：筛选 Vegfa-Flt1/Kdr，计算 simple_score，并输出与 evidence_score 的 Spearman 相关。
case_edges = None  # 保存目标配体—受体候选边
# ===== 项目06·任务2·需要完成（结束） =====
print(case_edges)  # 显示筛选结果


## 任务 3：细胞类型方向汇总

一条边同时包含发送类型和接收类型。按这两个字段汇总平均 `gnn_score` 和边数，绘制方向热图；行与列互换时含义也会互换。


In [ ]:
# ===== 项目06·任务3·需要完成（开始） =====
# TODO：按 src_cell_type 和 dst_cell_type 汇总平均分数与边数，并绘制热图。
communication = None  # 保存细胞类型方向汇总
# ===== 项目06·任务3·需要完成（结束） =====
print(communication)  # 显示汇总表


## 任务 4：空间候选关系与实验验证

绘制得分最高的局部 `Vegfa-Flt1/Kdr` 候选边，并查看训练曲线与独立遮边测试 Spearman。说明模型指标能支持的判断，以及仍需要哪些生物学验证。

配套遮边测试比较模型预测与被隐藏的 `evidence_score`：MAE 是平均绝对误差，MSE 是平均平方误差，Spearman 比较排序。Top 10% 重合召回分别取预测分数与参考分数最高的 `ceil(测试边数×0.1)` 条边，用交集数量除以该数量，表示优先候选名单的重合程度。


In [ ]:
# ===== 项目06·任务4·需要完成（开始） =====
# TODO：绘制空间候选边和训练曲线，并保存 result_summary.json。
result_summary = None  # 保存最终结果与解释边界
# ===== 项目06·任务4·需要完成（结束） =====
print(result_summary)  # 显示最终结果
